#### 매일 실행: 오늘자 raw.weather를 silver.farm_master의 활성 윈도우 농장들에 붙여 silver.farm_weather에 append

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import pandas as pd
import numpy as np

In [ ]:
# [1] raw.weather 파싱
# 주석행(#) 제거 → 공백 split → 컬럼 매핑 → 결측값(-9계열) null 처리

RAW_COLS = [
    "TM","STN","WS_AVG","WR_DAY","WD_MAX","WS_MAX","WS_MAX_TM","WD_INS","WS_INS","WS_INS_TM",
    "TA_AVG","TA_MAX","TA_MAX_TM","TA_MIN","TA_MIN_TM","TD_AVG","TS_AVG","TG_MIN",
    "HM_AVG","HM_MIN","HM_MIN_TM","PV_AVG","EV_S","EV_L","FG_DUR",
    "PA_AVG","PS_AVG","PS_MAX","PS_MAX_TM","PS_MIN","PS_MIN_TM",
    "CA_TOT","SS_DAY","SS_DUR","SS_CMB","SI_DAY","SI_60M_MAX","SI_60M_MAX_TM",
    "RN_DAY","RN_D99","RN_DUR","RN_60M_MAX","RN_60M_MAX_TM","RN_10M_MAX","RN_10M_MAX_TM",
    "RN_POW_MAX","RN_POW_MAX_TM","SD_NEW","SD_NEW_TM","SD_MAX","SD_MAX_TM",
    "TE_05","TE_10","TE_15","TE_30","TE_50"
]

# 시간 컬럼(HHMM 형식)은 string 유지, 나머지 double
STR_COLS = {
    "TM","WS_MAX_TM","WS_INS_TM","TA_MAX_TM","TA_MIN_TM","HM_MIN_TM",
    "PS_MAX_TM","PS_MIN_TM","SI_60M_MAX_TM","RN_60M_MAX_TM",
    "RN_10M_MAX_TM","RN_POW_MAX_TM","SD_NEW_TM","SD_MAX_TM"
}

tokens = F.split(F.col("#START7777"), r"\s+")

weather = (
    spark.table(f"{CATALOG}.raw.weather")
    .filter(~F.col("#START7777").startswith("#"))
    .select(
        *[
            F.when(F.split(F.col("#START7777"), r"\s+")[i].rlike(r"^-9+\.?0*$"), None)
             .otherwise(F.split(F.col("#START7777"), r"\s+")[i].cast("string" if col in STR_COLS else DoubleType()))
             .alias(col)
            for i, col in enumerate(RAW_COLS)
        ]
    )
    .withColumn("STN", F.col("STN").cast("int"))
    .withColumn("TM", F.to_date(F.col("TM"), "yyyyMMdd"))
)


In [ ]:
# [2] farm_master ↔ STN 최근접 매칭 (Haversine, driver 실행)

stn_pdf = pd.read_csv("/Volumes/dt4_team1_databricks/raw/test/stn_locations.csv")

farm_pdf = (
    spark.table(f"{CATALOG}.silver.farm_master")
    .select("farm_id", "latitude", "longitude")
    .dropDuplicates(["farm_id"])
    .toPandas()
)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def match_stn(row):
    dists = haversine(row.latitude, row.longitude, stn_pdf["LAT"].values, stn_pdf["LON"].values)
    idx = np.argmin(dists)
    return int(stn_pdf.iloc[idx]["STN"]), stn_pdf.iloc[idx]["STN_KO"], round(float(dists[idx]), 3)

farm_pdf[["matched_STN", "matched_STN_KO", "matched_dist_km"]] = pd.DataFrame(
    farm_pdf.apply(match_stn, axis=1).tolist(), index=farm_pdf.index
)

farm_stn = spark.createDataFrame(farm_pdf[["farm_id", "matched_STN", "matched_STN_KO", "matched_dist_km"]])


In [ ]:
# [3] farm_master × STN매칭 × 날씨 조인
# TEST_DATE: 테스트 기준일 | 운영 시 F.current_date() 로 교체

BASE_DATE = "2026-06-30"
# BASE_DATE = F.current_date()  # 운영 기준일

weather_renamed = weather.withColumnRenamed("STN", "matched_STN")
result = (
    spark.table(f"{CATALOG}.silver.farm_master")
    .withColumnRenamed("address", "farm_address")
    .join(farm_stn, "farm_id")
    .join(weather_renamed, "matched_STN")
    .filter(
        (F.datediff(F.lit(BASE_DATE).cast("date"), F.col("TM")) >= 0) &
        (F.datediff(F.lit(BASE_DATE).cast("date"), F.col("TM")) <= 6)
    )
)

In [ ]:
# [4] silver.farm_weather 저장
# TM 파티션 단위 overwrite → 날짜별 독립 적재

# 7일 초과 삭제
spark.sql(f"DELETE FROM {CATALOG}.silver.farm_weather WHERE TM < date_sub('{BASE_DATE}', 6)")


# 당일치 append
(
    result.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOG}.silver.farm_weather")
)

print(f"✅ silver.farm_weather 저장 완료: {result.count()} rows")

In [ ]:
%sql
select distinct(TM) from dt4_team1_databricks.silver.farm_weather 